# 15 — Data hipotetik berlabel (semua skenario) + ANN-LM asli tanpa entropi

**Permintaan:** dosen Bu Luh meminta *data hipotetik yang sudah berlabel untuk
semua skenario* beserta programnya, untuk membangun **sistem notifikasi
gangguan** sensor kelembaban tanah.

**Basis:** program pertama yang memakai data buatan
(`final_databaru_optimized.ipynb`, Des 2025): data `tabel_sensor4_generated.csv`,
lima skenario fault, fitur lag per sampel, ANN 1 dan 2 layer, **tanpa entropi**.

**Yang diperbarui dibanding program pertama**

| Bagian | Program pertama (Des 2025) | Notebook ini |
|---|---|---|
| Pelatihan ANN | `scipy.optimize.least_squares(method='lm')` — MINPACK, Jacobian numerik, init `randn`, 1 keluaran sigmoid | **LM asli** notebook 10/11/14: langkah Gauss-Newton terdamping + rasio gain (Nielsen), **Jacobian analitik** (lolos self-check), init Xavier, 2 keluaran one-hot + softmax |
| Simulator fault | drift 0,15 poin/**sampel** (tembus > 100 %), bias konstan 0,35, spike 0,3·std berkala | versi notebook 14 tingkat "sedang": **bias episode 5 poin**, **drift linear total 5 poin per episode**, **spike relatif 30 %** (kepadatan 1,52 %), hardware malfunction (macet 8 %, hilang 5 %); 3 episode × 3 hari lalu kembali normal; semua dijaga 0–100 % |
| Bentuk data | 4 sensor dirata-rata jadi 1 sinyal | **dua versi**: 4 sensor terpisah (label per sensor) **dan** 1 sinyal rata-rata |
| Skenario | S1..S5, sebagian kondisi dipilih acak | **S1..S5 lengkap = 16 kondisi fault** + normal, seed tetap (bisa direproduksi) |
| Pembagian data | 60/20/20 berurutan waktu | blok harian bergilir 3:1:1 (lihat §4 — dengan 60/20/20 berurutan, bagian uji tidak berisi episode drift/bias sama sekali) |
| Entropi | tidak dipakai | tetap **tidak dipakai** |

**Isi notebook**

1. §1 Data sumber + QC
2. §2 Simulator fault + peta skenario
3. §3 **Data hipotetik berlabel** (deliverable utama) → `exports/15_data_hipotetik_*.csv.gz` + kamus kolom
4. §4 Fitur & pembagian data
5. §5 ANN-LM asli (self-check Jacobian, lalu latih per kondisi)
6. §6 Hasil + prediksi per sampel (probabilitas & alarm) untuk sistem notifikasi


In [ ]:
import os, time, json, io
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "2")
import time as _time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from IPython.display import FileLink, display

NOTEBOOK_START = time.time()
SMOKE = os.environ.get("SMOKE", "0") == "1"      # uji cepat lokal: data dipotong, iterasi sedikit
RANDOM_SEED = 42
EXPORT_DIR = Path("exports"); EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def export_df(df, name, gz=False, index=False):
    p = EXPORT_DIR / (f"{name}.csv.gz" if gz else f"{name}.csv")
    df.to_csv(p, index=index)
    display(FileLink(str(p)))
    return p

print("SMOKE =", SMOKE)


# §1 — Data sumber

`tabel_sensor4_generated.csv` — data yang sama dengan program pertama: 4 sensor
kelembaban tanah (`kelembaban1..4`, %), interval ±30 detik, 2025-09-14 s.d.
2025-10-17. Satu baris berstempel 1970 (jam perangkat belum sinkron) dibuang;
baris lain tidak diubah. Sinyal ini dianggap **normal** (belum ada fault), lalu
fault disuntikkan di §3.


In [ ]:
DATA_NAME = "tabel_sensor4_generated.csv"
REPO_RAW = "https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/tabel_sensor4_generated.csv"
SENSORS = ["kelembaban1", "kelembaban2", "kelembaban3", "kelembaban4"]

def load_data():
    for p in [Path(DATA_NAME), *Path("/kaggle/input").glob(f"**/{DATA_NAME}")]:
        if p.exists():
            return pd.read_csv(p)
    import requests
    r = requests.get(REPO_RAW, timeout=120); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

raw = load_data()
raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True, errors="coerce")
n0 = len(raw)
bad_ts = raw["timestamp"].isna() | (raw["timestamp"] < pd.Timestamp("2020-01-01", tz="UTC"))
df = raw.loc[~bad_ts, ["timestamp", *SENSORS]].sort_values("timestamp").reset_index(drop=True)
if SMOKE:
    df = df.iloc[:20000].reset_index(drop=True)
dt = df["timestamp"].diff().dt.total_seconds()
qc = pd.DataFrame([{
    "baris_awal": n0, "dibuang_stempel_invalid": int(bad_ts.sum()), "baris_dipakai": len(df),
    "mulai": df["timestamp"].iloc[0], "selesai": df["timestamp"].iloc[-1],
    "interval_median_s": float(dt.median()), "NaN_sensor": int(df[SENSORS].isna().sum().sum()),
}])
display(qc)
display(df[SENSORS].describe().round(2))
assert df[SENSORS].isna().sum().sum() == 0
SAMPLING_SECONDS = 30


# §2 — Simulator fault (versi notebook 14, tingkat "sedang")

| Fault | Bentuk | Parameter |
|---|---|---|
| **drift** | naik linear dari 0 ke 5 poin selama satu episode 3 hari, lalu reset (kalibrasi ulang) | 3 episode, mulai di 10 %, 40 %, 70 % panjang data |
| **bias** | offset +5 poin selama episode, lalu kembali normal | episode sama dengan drift |
| **spike** | `x(t)·(1 + 0,30)` di posisi acak | 1,52 % titik |
| **hardware** | nilai macet (diganti nilai acak lain dari seri) 8 % titik, data hilang (NaN) 5 % titik | — |
| **kombinasi** | semua komponen disuntik pada interval episode yang **sama** | — |

Label per sampel disimpan sebagai **bitmask**: drift = 1, spike = 2, bias = 4,
hardware = 8. Contoh: 6 = spike + bias. Kolom `jenis_*` di data ekspor
menuliskannya sebagai teks.

**Skenario** mengikuti program pertama:

| Skenario | Isi | Jumlah kondisi |
|---|---|---|
| S1 | satu fault, jenisnya dipilih acak (seed tetap) | 1 |
| S2 | fault tunggal: drift, spike, bias, hardware | 4 |
| S3 | kombinasi dua fault | 6 |
| S4 | kombinasi tiga fault | 4 |
| S5 | keempat fault sekaligus | 1 |


In [ ]:
DRIFT_EPISODE_DAYS = 3.0
DRIFT_N_EPISODES = 3
DRIFT_SAMPLE_SECONDS = SAMPLING_SECONDS
DRIFT_TOTAL_POINTS = float(os.environ.get("DRIFT_TOTAL_POINTS", 5.0))
BIAS_EPISODE_POINTS = float(os.environ.get("BIAS_EPISODE_POINTS", 5.0))
SPIKE_REL = float(os.environ.get("SPIKE_REL", 0.30))
SPIKE_DENSITY = 0.0152

def _episode_mask(n, days=DRIFT_EPISODE_DAYS, n_episodes=DRIFT_N_EPISODES):
    # Episode identik notebook 11: mulai di 10%, 40%, 70% deret, panjang 3 hari.
    ep_len = min(n, max(1, int(round(days * 86400.0 / DRIFT_SAMPLE_SECONDS))))
    mask = np.zeros(n, bool); ramp = np.zeros(n)
    for s in (np.linspace(0.10, 0.70, n_episodes) * n).astype(int):
        e = min(n, s + ep_len)
        mask[s:e] = True
        ramp[s:e] = np.arange(1, e - s + 1) / ep_len
    return mask, ramp

FAULT_BIT = {"drift": 1, "spike": 2, "bias": 4, "hardware": 8}

def simulate_drift_fault(x, total=DRIFT_TOTAL_POINTS, seed=None, within=None):
    # Linear per episode: naik dari 0 ke `total` poin selama 3 hari, lalu reset.
    ep, ramp = _episode_mask(len(x))
    return x + total * ramp, ep.astype(np.uint8) * FAULT_BIT["drift"]

def simulate_spike_fault(x, k=SPIKE_REL, density=SPIKE_DENSITY, seed=None, within=None):
    # Spike relatif x(t)*(1+k) di posisi acak; kepadatan dihitung atas kandidat posisi.
    rng = np.random.default_rng(None if seed is None else seed + 1)
    cand = np.arange(len(x)) if within is None else np.flatnonzero(within)
    n_sp = max(1, int(round(density * len(cand))))
    pos = rng.choice(cand, size=min(n_sp, len(cand)), replace=False)
    y = x.copy(); y[pos] = x[pos] * (1.0 + k)
    mask = np.zeros(len(x), np.uint8); mask[pos] = FAULT_BIT["spike"]
    return y, mask

def simulate_bias_fault(x, bias=BIAS_EPISODE_POINTS, seed=None, within=None):
    # Bias episode: offset selama episode, lalu kembali normal.
    ep, _ = _episode_mask(len(x))
    return x + bias * ep, ep.astype(np.uint8) * FAULT_BIT["bias"]

def simulate_hardware_fault(x, stuck_prob=0.08, loss_prob=0.05, seed=None, within=None):
    rng = np.random.default_rng(None if seed is None else seed + 2)
    n = len(x); rv = rng.random(n); idx = rng.integers(n, size=n)
    allow = np.ones(n, bool) if within is None else within
    m1 = (rv < stuck_prob) & allow; y = x.copy(); y[m1] = x[idx[m1]]
    m2 = (rv < loss_prob) & allow; y[m2] = np.nan
    return y, (m1 | m2).astype(np.uint8) * FAULT_BIT["hardware"]

def simulate_multiple_faults(x, faults, seed=None):
    # Kombinasi (>= 2 komponen): semua komponen pada interval episode yang sama.
    within = _episode_mask(len(x))[0] if len(faults) > 1 else None
    y = x.copy(); m_ = np.zeros(len(x), np.uint8)
    for f, kw in faults:
        y, mi = f(y, **kw, seed=seed, within=within); m_ |= mi
    # Jaga domain fisik: injeksi tidak boleh keluar 0-100 %.
    # m_ = bitmask komponen per sampel (FAULT_BIT), bukan bool.
    return np.clip(y, 0.0, 100.0), m_

def simulate_choose_one(x, options, seed=None, within=None):
    rng = np.random.default_rng(seed)
    f, kw = options[rng.integers(len(options))]
    y, m = f(x, **kw, seed=seed, within=within)
    return np.clip(y, 0.0, 100.0), m

_D = {"total": DRIFT_TOTAL_POINTS}
_S = {"k": SPIKE_REL}
_B = {"bias": BIAS_EPISODE_POINTS}
_H = {"stuck_prob": 0.08, "loss_prob": 0.05}

CONDITIONS = {
    "faulty": ("S1", [(simulate_choose_one, {"options": [(simulate_drift_fault, _D), (simulate_spike_fault, _S),
                                                          (simulate_bias_fault, _B), (simulate_hardware_fault, _H)]})]),
    "drift": ("S2", [(simulate_drift_fault, _D)]),
    "spike": ("S2", [(simulate_spike_fault, _S)]),
    "bias": ("S2", [(simulate_bias_fault, _B)]),
    "hardware": ("S2", [(simulate_hardware_fault, _H)]),
    "bias+malfunc": ("S3", [(simulate_bias_fault, _B), (simulate_hardware_fault, _H)]),
    "spike+malfunc": ("S3", [(simulate_spike_fault, _S), (simulate_hardware_fault, _H)]),
    "spike+bias": ("S3", [(simulate_spike_fault, _S), (simulate_bias_fault, _B)]),
    "drift+malfunc": ("S3", [(simulate_drift_fault, _D), (simulate_hardware_fault, _H)]),
    "drift+bias": ("S3", [(simulate_drift_fault, _D), (simulate_bias_fault, _B)]),
    "drift+spike": ("S3", [(simulate_drift_fault, _D), (simulate_spike_fault, _S)]),
    "spike+bias+malfunc": ("S4", [(simulate_spike_fault, _S), (simulate_bias_fault, _B), (simulate_hardware_fault, _H)]),
    "drift+bias+malfunc": ("S4", [(simulate_drift_fault, _D), (simulate_bias_fault, _B), (simulate_hardware_fault, _H)]),
    "spike+drift+malfunc": ("S4", [(simulate_spike_fault, _S), (simulate_drift_fault, _D), (simulate_hardware_fault, _H)]),
    "drift+spike+bias": ("S4", [(simulate_drift_fault, _D), (simulate_spike_fault, _S), (simulate_bias_fault, _B)]),
    "spike+bias+malfunc+drift": ("S5", [(simulate_spike_fault, _S), (simulate_bias_fault, _B),
                                         (simulate_hardware_fault, _H), (simulate_drift_fault, _D)]),
}
SCENARIO_ORDER = ["S1", "S2", "S3", "S4", "S5"]
assert len(CONDITIONS) == 16

def bits_to_text(b):
    b = int(b)
    if b == 0:
        return "normal"
    return "+".join(n for n, v in FAULT_BIT.items() if b & v)

print(pd.Series({k: v[0] for k, v in CONDITIONS.items()}).value_counts().sort_index())


# §3 — Data hipotetik berlabel (deliverable utama)

Untuk **tiap kondisi** fault dibuat satu salinan data penuh dengan fault
disuntikkan, seed tetap (hasil selalu sama tiap dijalankan).

**Versi A — 4 sensor terpisah.** Fault hanya masuk ke sebagian sensor
(1–4 sensor, bergiliran antar kondisi, sensor mana dipilih acak dengan seed
tetap). Sensor lain tetap bersih — seperti kondisi lapangan, tidak semua
sensor rusak bersamaan.

**Versi B — 1 sinyal rata-rata** (seperti program pertama): rata-rata 4 sensor
bersih → `kelembaban_tanah_persen`, lalu fault disuntikkan ke sinyal itu.

Nilai yang hilang karena hardware malfunction **dibiarkan NaN** di kolom data
(itu bagian dari gejala gangguan yang harus dikenali sistem notifikasi);
nilai bersihnya tetap ada di kolom `*_asli`.

Satu file per skenario: `15_data_hipotetik_4sensor_S1..S5.csv.gz`,
`15_data_hipotetik_rata2_S1..S5.csv.gz`, plus `..._S0_normal` (data bersih
tanpa fault) dan `15_kamus_kolom.csv`.


In [ ]:
X_clean = df[SENSORS].to_numpy(dtype=float)
avg_clean = X_clean.mean(axis=1)
T = len(df)
SUBSET_SIZES = [1, 2, 3, 4]

GEN_4S, GEN_AVG = {}, {}     # kondisi -> dict array
rng_master = np.random.default_rng(RANDOM_SEED)
for ci, (cond, (scen, faults)) in enumerate(CONDITIONS.items()):
    seed = int(rng_master.integers(1_000_000_000))
    rng = np.random.default_rng(seed)
    # --- versi A: 4 sensor ---
    k = SUBSET_SIZES[ci % len(SUBSET_SIZES)]
    subset = np.sort(rng.choice(4, size=k, replace=False))
    Y = X_clean.copy(); M = np.zeros((T, 4), dtype=np.uint8)
    for s in subset:
        y, m = simulate_multiple_faults(X_clean[:, s].copy(), faults, seed=int(rng.integers(1_000_000_000)))
        Y[:, s] = y; M[:, s] = m
    GEN_4S[cond] = {"scen": scen, "Y": Y, "M": M, "subset": subset}
    # --- versi B: rata-rata ---
    y, m = simulate_multiple_faults(avg_clean.copy(), faults, seed=seed + 7)
    GEN_AVG[cond] = {"scen": scen, "y": y, "m": m}

def frame_4s(cond):
    g = GEN_4S[cond]
    out = pd.DataFrame({"timestamp": df["timestamp"], "skenario": g["scen"], "kondisi": cond})
    for s, name in enumerate(SENSORS):
        out[name] = np.round(g["Y"][:, s], 4)
    for s, name in enumerate(SENSORS):
        out[f"{name}_asli"] = np.round(X_clean[:, s], 4)
    for s in range(4):
        out[f"label_s{s+1}"] = (g["M"][:, s] > 0).astype(np.int8)
    for s in range(4):
        out[f"jenis_s{s+1}"] = pd.Series(g["M"][:, s]).map(bits_to_text).to_numpy()
    lab = g["M"] > 0
    out["label_fault"] = lab.any(axis=1).astype(np.int8)
    out["sensor_rusak"] = [",".join(f"s{s + 1}" for s in np.flatnonzero(r)) for r in lab]
    out["sensor_terinjeksi"] = ",".join(f"s{s + 1}" for s in g["subset"])
    return out

def frame_avg(cond):
    g = GEN_AVG[cond]
    return pd.DataFrame({"timestamp": df["timestamp"], "skenario": g["scen"], "kondisi": cond,
                         "kelembaban_tanah_persen": np.round(g["y"], 4),
                         "kelembaban_tanah_persen_asli": np.round(avg_clean, 4),
                         "label_fault": (g["m"] > 0).astype(np.int8),
                         "jenis_fault": pd.Series(g["m"]).map(bits_to_text).to_numpy()})

# --- asersi konsistensi (sumber: definisi simulator notebook 14) ---
for cond in CONDITIONS:
    g = GEN_4S[cond]
    changed = ~np.isclose(g["Y"], X_clean, equal_nan=False)          # NaN dihitung berubah
    assert not (changed & (g["M"] == 0)).any(), f"{cond}: nilai berubah tanpa label"
    assert ((g["M"] > 0).sum() > 0) and ((g["M"] == 0).sum() > 0), cond
    v = g["Y"][~np.isnan(g["Y"])]; assert v.min() >= 0 and v.max() <= 100, cond
    untouched = np.setdiff1d(np.arange(4), g["subset"])
    assert (g["M"][:, untouched] == 0).all(), f"{cond}: sensor tak terinjeksi berlabel"
    ga = GEN_AVG[cond]
    ch = ~np.isclose(ga["y"], avg_clean)
    assert not (ch & (ga["m"] == 0)).any(), f"{cond}: rata2 berubah tanpa label"
    assert 0 < (ga["m"] > 0).sum() < T, cond
print("Asersi label lolos untuk 16 kondisi x 2 versi.")

# --- ekspor per skenario ---
normal_4s = pd.DataFrame({"timestamp": df["timestamp"], "skenario": "S0", "kondisi": "normal",
                          **{n: X_clean[:, s] for s, n in enumerate(SENSORS)},
                          **{f"label_s{s+1}": np.zeros(T, np.int8) for s in range(4)},
                          "label_fault": np.zeros(T, np.int8)})
export_df(normal_4s, "15_data_hipotetik_4sensor_S0_normal", gz=True)
export_df(pd.DataFrame({"timestamp": df["timestamp"], "skenario": "S0", "kondisi": "normal",
                        "kelembaban_tanah_persen": avg_clean, "label_fault": np.zeros(T, np.int8)}),
          "15_data_hipotetik_rata2_S0_normal", gz=True)
for scen in SCENARIO_ORDER:
    conds = [c for c, (s, _) in CONDITIONS.items() if s == scen]
    export_df(pd.concat([frame_4s(c) for c in conds], ignore_index=True),
              f"15_data_hipotetik_4sensor_{scen}", gz=True)
    export_df(pd.concat([frame_avg(c) for c in conds], ignore_index=True),
              f"15_data_hipotetik_rata2_{scen}", gz=True)

# --- ringkasan label ---
rows = []
for cond, (scen, _) in CONDITIONS.items():
    g = GEN_4S[cond]; ga = GEN_AVG[cond]
    r = {"skenario": scen, "kondisi": cond, "sensor_terinjeksi": ",".join(f"s{s + 1}" for s in g["subset"]),
         "n_sampel": T}
    for s in range(4):
        r[f"fault_s{s+1}_%"] = round(100 * float((g["M"][:, s] > 0).mean()), 2)
    r["fault_any_4sensor_%"] = round(100 * float((g["M"] > 0).any(axis=1).mean()), 2)
    r["NaN_4sensor"] = int(np.isnan(g["Y"]).sum())
    r["fault_rata2_%"] = round(100 * float((ga["m"] > 0).mean()), 2)
    r["jenis_rata2_terbanyak"] = pd.Series(ga["m"][ga["m"] > 0]).map(bits_to_text).value_counts().index[0]
    rows.append(r)
ringkasan_label = pd.DataFrame(rows)
export_df(ringkasan_label, "15_ringkasan_label")
display(ringkasan_label)


In [ ]:
KAMUS = [
    ("timestamp", "waktu sampel (UTC), interval ±30 detik", "keduanya"),
    ("skenario", "S0 normal; S1 satu fault acak; S2 fault tunggal; S3 dua fault; S4 tiga fault; S5 empat fault", "keduanya"),
    ("kondisi", "nama kondisi fault yang disuntikkan pada salinan data ini (16 kondisi + normal)", "keduanya"),
    ("kelembaban1..4", "nilai sensor SETELAH fault disuntikkan (%); NaN = data hilang akibat hardware malfunction", "4 sensor"),
    ("kelembaban1..4_asli", "nilai sensor sebelum fault (bersih)", "4 sensor"),
    ("label_s1..s4", "1 = sampel sensor itu terkena fault, 0 = normal", "4 sensor"),
    ("jenis_s1..s4", "jenis fault pada sampel itu: normal / drift / spike / bias / hardware / gabungannya, mis. 'spike+bias'", "4 sensor"),
    ("label_fault", "1 = minimal satu sensor (atau sinyal rata-rata) terkena fault pada sampel itu", "keduanya"),
    ("sensor_rusak", "daftar sensor yang berlabel fault pada sampel itu, mis. 's1,s3'", "4 sensor"),
    ("sensor_terinjeksi", "sensor yang disuntik fault pada kondisi ini (konstan per kondisi)", "4 sensor"),
    ("kelembaban_tanah_persen", "rata-rata 4 sensor SETELAH fault disuntikkan ke sinyal rata-rata (%)", "rata-rata"),
    ("kelembaban_tanah_persen_asli", "rata-rata 4 sensor bersih", "rata-rata"),
    ("jenis_fault", "jenis fault pada sampel sinyal rata-rata", "rata-rata"),
    ("kode bitmask", "drift=1, spike=2, bias=4, hardware=8 (dipakai di dalam program; di CSV sudah ditulis sebagai teks)", "—"),
    ("parameter fault", f"drift total {DRIFT_TOTAL_POINTS} poin/episode, bias {BIAS_EPISODE_POINTS} poin/episode, "
                        f"spike x(1+{SPIKE_REL}) pada {SPIKE_DENSITY*100:.2f}% titik, hardware macet 8% + hilang 5%, "
                        f"{DRIFT_N_EPISODES} episode x {DRIFT_EPISODE_DAYS:g} hari", "—"),
]
kamus = pd.DataFrame(KAMUS, columns=["kolom", "arti", "versi"])
export_df(kamus, "15_kamus_kolom")
display(kamus)


In [ ]:
# Contoh sinyal: asli vs setelah fault, titik berlabel fault ditandai
BLOCK_DAY = int(round(86400 / SAMPLING_SECONDS))

def plot_conditions(version):
    fig, axes = plt.subplots(4, 4, figsize=(20, 13), sharex=True)
    for ax, cond in zip(axes.ravel(), CONDITIONS):
        if version == "4sensor":
            g = GEN_4S[cond]; s = int(g["subset"][0])
            y, base, m = g["Y"][:, s], X_clean[:, s], g["M"][:, s]
            ttl = f"{CONDITIONS[cond][0]} {cond} (sensor {s+1})"
        else:
            g = GEN_AVG[cond]; y, base, m = g["y"], avg_clean, g["m"]
            ttl = f"{CONDITIONS[cond][0]} {cond} (rata-rata)"
        # zoom 2 hari di sekitar awal episode pertama supaya spike/hardware terlihat jelas
        e0 = int(0.10 * T); a, b = max(0, e0 - BLOCK_DAY // 2), min(T, e0 + 3 * BLOCK_DAY // 2)
        t = np.arange(a, b)
        ax.plot(t, base[a:b], lw=0.7, color="0.6", label="asli")
        ax.plot(t, y[a:b], lw=0.7, color="tab:blue", label="setelah fault")
        idx = a + np.flatnonzero(m[a:b] > 0)
        ax.scatter(idx, np.nan_to_num(y[idx], nan=base[idx]), s=10, color="tab:red", label="label fault", zorder=3)
        ax.set_title(ttl, fontsize=9)
    axes[0, 0].legend(fontsize=7, markerscale=6)
    fig.suptitle(f"Data hipotetik berlabel — versi {version} (zoom 2 hari di sekitar awal episode pertama; NaN = celah)", fontsize=13)
    fig.tight_layout()
    p = EXPORT_DIR / f"15_contoh_sinyal_{version}.png"; fig.savefig(p, dpi=110); plt.show()

plot_conditions("4sensor")
plot_conditions("rata2")


# §4 — Fitur & pembagian data (alur program pertama)

**Fitur per sampel, tanpa entropi** (sama seperti program pertama): nilai sinyal
10 langkah sebelumnya (`lag1..lag10`) + selisih terhadap sampel sebelumnya
(`diff`). Karena data hilang (NaN) dibiarkan apa adanya, nilai yang hilang
diisi nilai terakhir yang valid, dan ditambah dua penanda: `isna` (sampel ini
hilang) dan `isna_10` (berapa sampel hilang di 10 langkah terakhir) — setara
kolom `_isna` di program pertama.

**Pembagian data.** Program pertama memakai 60/20/20 berurutan waktu. Dengan
simulator baru, episode drift/bias ada di 10 %, 40 %, 70 % data, sehingga
bagian uji (80–100 %) **tidak berisi episode drift/bias sama sekali** dan
evaluasinya tidak bermakna. Jadi data dipotong per **blok 1 hari** dan blok
dibagi bergiliran latih, latih, latih, validasi, uji (3:1:1 ≈ 60/20/20). Tiap
episode 3 hari kebagian ke beberapa bagian, dan blok tetap utuh (tidak
diacak per sampel) sehingga sampel bertetangga tidak bocor antar bagian
kecuali 10 lag di batas blok.

`StandardScaler` hanya di-*fit* di data latih; ambang probabilitas dipilih
yang memaksimalkan F1 di validasi (seperti program pertama), lalu dilaporkan
di data uji.

**Versi 4 sensor:** satu model untuk semua sensor — baris tiap sensor
ditumpuk, fitur hanya dari sensor itu sendiri. Hasil dilaporkan total dan per
sensor.


In [ ]:
LAG_K = 10
BLOCK_LEN = int(round(86400 / SAMPLING_SECONDS))       # 1 hari
SPLIT_PATTERN = np.array([0, 0, 0, 1, 2])               # 0 latih, 1 validasi, 2 uji
SPLIT_NAME = {0: "latih", 1: "validasi", 2: "uji"}
split_of = SPLIT_PATTERN[(np.arange(T) // BLOCK_LEN) % len(SPLIT_PATTERN)]
valid_row = np.arange(T) >= LAG_K                        # baris awal tanpa lag lengkap dibuang
print(pd.Series(split_of[valid_row]).map(SPLIT_NAME).value_counts())

FEATURE_NAMES = [f"lag{k}" for k in range(1, LAG_K + 1)] + ["diff", "isna", "isna_10"]

def lag_features(x):
    x = np.asarray(x, dtype=float)
    na = np.isnan(x)
    xf = pd.Series(x).ffill().bfill().to_numpy()
    cols = [np.roll(xf, k) for k in range(1, LAG_K + 1)]
    cols.append(xf - np.roll(xf, 1))
    cols.append(na.astype(float))
    cols.append(pd.Series(na.astype(float)).rolling(LAG_K, min_periods=1).sum().shift(1).fillna(0).to_numpy())
    return np.column_stack(cols)       # baris < LAG_K dibuang lewat valid_row

assert lag_features(np.arange(20.0)).shape == (20, len(FEATURE_NAMES))


# §5 — ANN-LM asli

Kode Levenberg–Marquardt di bawah **identik** dengan notebook 10/11/14
(diambil langsung dari builder notebook 11). Ringkasnya, tiap iterasi:

1. hitung residual `r = keluaran − target` dan **Jacobian analitik** `J`
   (backpropagation per keluaran, bukan selisih hingga);
2. selesaikan `(JᵀJ + λ·diag(JᵀJ))·Δ = −Jᵀr`;
3. kalau galat turun, terima langkah dan kecilkan λ menurut rasio gain
   (aturan Nielsen); kalau naik, tolak dan besarkan λ.

Keluaran 2 unit (normal, fault) dengan target one-hot; probabilitas fault =
softmax unit kedua. Arsitektur mengikuti program pertama: **1 layer (6)** dan
**2 layer (6, 3)**, aktivasi `tanh`.


In [ ]:
import time as _time

def lm_init_layers(n_in, hidden, n_out, seed=0):
    rng = np.random.default_rng(seed)
    dims = [n_in, *hidden, n_out]
    layers = []
    for i in range(len(dims) - 1):
        lim = np.sqrt(6.0 / (dims[i] + dims[i + 1]))   # Xavier, dipakai untuk tanh & relu
        layers.append([rng.uniform(-lim, lim, (dims[i], dims[i + 1])), np.zeros(dims[i + 1])])
    return layers

def lm_pack(layers):
    return np.concatenate([W.ravel() for W, _ in layers] + [b for _, b in layers])

def lm_unpack(theta, template):
    out, i = [], 0
    for W, b in template:
        out.append([theta[i:i + W.size].reshape(W.shape), theta[i + W.size:i + W.size + b.size]])
        i += W.size + b.size
    return out

def lm_forward(layers, X, activation="tanh"):
    acts = [X]
    for li, (W, b) in enumerate(layers):
        z = acts[-1] @ W + b
        if li == len(layers) - 1:
            acts.append(z)                              # keluaran linear (target one-hot)
        elif activation == "relu":
            acts.append(np.maximum(z, 0.0))
        else:
            acts.append(np.tanh(z))
    return acts[-1], acts

def lm_residuals(theta, template, X, Y, activation="tanh"):
    out, _ = lm_forward(lm_unpack(theta, template), X, activation)
    return (out - Y).ravel()

def lm_residual_jacobian(theta, template, X, Y, activation="tanh", mode="analytic"):
    layers = lm_unpack(theta, template)
    out, acts = lm_forward(layers, X, activation)
    m, n_out = out.shape
    if mode == "numeric":
        r0 = (out - Y).ravel()
        J = np.empty((r0.size, theta.size))
        eps = 1e-6
        for p in range(theta.size):
            tp = theta.copy(); tp[p] += eps
            op, _ = lm_forward(lm_unpack(tp, template), X, activation)
            J[:, p] = ((op - Y).ravel() - r0) / eps
        return J
    L = len(layers)
    delta = np.broadcast_to(np.eye(n_out), (m, n_out, n_out)).astype(float)
    blocks = [None] * L
    for l in range(L - 1, -1, -1):
        W, b = layers[l]
        a_prev = acts[l]
        n_in, n_out_l = W.shape
        dW = np.einsum("ip,icq->icpq", a_prev, delta, optimize=True)
        blocks[l] = (dW.reshape(m * n_out, n_in * n_out_l),
                     delta.reshape(m * n_out, n_out_l))
        if l > 0:
            back = np.einsum("icq,pq->icp", delta, W, optimize=True)
            h = acts[l]          # l>=1 -> acts[l] selalu keluaran hidden layer
            deriv = (h > 0).astype(float) if activation == "relu" else (1.0 - h ** 2)
            delta = back * deriv[:, None, :]
    J = np.empty((m * n_out, theta.size))
    col = 0
    for dW, db in blocks:
        J[:, col:col + dW.shape[1]] = dW; col += dW.shape[1]
        J[:, col:col + db.shape[1]] = db; col += db.shape[1]
    return J

LM_JAC_MODE = "analytic"

def lm_jac(theta, template, X, Y, activation="tanh"):
    return lm_residual_jacobian(theta, template, X, Y, activation, mode=LM_JAC_MODE)

def lm_fit(X, Y, n_in, hidden, n_out, activation="tanh", iters=40, seed=0,
           lam0=1e-3, max_train_s=60.0):
    template = lm_init_layers(n_in, hidden, n_out, seed)
    theta = lm_pack(template)
    lam = lam0
    hist = {"loss": [], "grad": [], "lam": [], "nfev": 0, "iters": 0, "status": "max_iter"}
    r = lm_residuals(theta, template, X, Y, activation); hist["nfev"] += 1
    E = 0.5 * float(r @ r)
    _t0 = _time.perf_counter()
    for k in range(iters):
        if _time.perf_counter() - _t0 > max_train_s:
            hist["status"] = "budget(MAX_TRAIN_S)"; break
        J = lm_jac(theta, template, X, Y, activation)
        g = J.T @ r
        A = J.T @ J
        gn = float(np.linalg.norm(g))
        hist["loss"].append(float(np.mean(r * r))); hist["grad"].append(gn); hist["lam"].append(float(lam))
        hist["iters"] = k + 1
        if gn < 1e-10:
            hist["status"] = "converged(grad)"; break
        d = np.diag(A).copy(); d = np.where(d < 1e-12, 1e-12, d)
        try:
            step = np.linalg.solve(A + lam * np.diag(d), -g)
        except np.linalg.LinAlgError:
            lam *= 10.0; continue
        if not np.all(np.isfinite(step)):
            lam *= 10.0; continue
        theta_new = theta + step
        r_new = lm_residuals(theta_new, template, X, Y, activation); hist["nfev"] += 1
        E_new = 0.5 * float(r_new @ r_new)
        denom = 0.5 * float(step @ (lam * d * step - g))
        rho = (E - E_new) / (denom if abs(denom) > 1e-300 else 1e-300)
        if E_new < E:
            if abs(E - E_new) <= 1e-12 * (1.0 + abs(E)):
                theta, r, E = theta_new, r_new, E_new
                hist["loss"].append(float(np.mean(r * r))); hist["lam"].append(float(lam))
                hist["status"] = "converged(tol)"; break
            theta, r, E = theta_new, r_new, E_new
            lam = lam * max(1.0 / 3.0, 1.0 - (2.0 * rho - 1.0) ** 3) if rho > 0 else lam * 2.0
        else:
            lam *= 2.0
        lam = float(np.clip(lam, 1e-12, 1e12))
    if len(hist["loss"]) == 0:
        hist["loss"].append(float(np.mean(r * r)))
    return {"theta": theta, "template": template, "hist": hist,
            "train_mse": float(np.mean(lm_residuals(theta, template, X, Y, activation) ** 2))}

def lm_predict_proba(theta, template, X, activation="tanh"):
    out, _ = lm_forward(lm_unpack(theta, template), np.asarray(X, dtype=float), activation)
    e = out - out.max(axis=1, keepdims=True)
    P = np.exp(e); P /= P.sum(axis=1, keepdims=True)
    return P


# --- self-check: Jacobian analitik vs numerik ---
_rng = np.random.default_rng(7)
for _hidden in ([6], [6, 3]):
    _tpl = lm_init_layers(len(FEATURE_NAMES), _hidden, 2, seed=7)
    _X = _rng.normal(size=(16, len(FEATURE_NAMES))); _Y = np.eye(2)[_rng.integers(0, 2, 16)]
    _th = lm_pack(_tpl)
    _Ja = lm_residual_jacobian(_th, _tpl, _X, _Y, "tanh", mode="analytic")
    _Jn = lm_residual_jacobian(_th, _tpl, _X, _Y, "tanh", mode="numeric")
    _err = float(np.max(np.abs(_Ja - _Jn) / np.maximum(1e-8, np.abs(_Jn))))
    print(f"hidden={_hidden}: max |rel err| Jacobian = {_err:.2e}")
    assert _err < 1e-3, "Jacobian analitik tidak cocok"
print("Jacobian analitik LOLOS self-check.")


In [ ]:
ARCHS = {"1 layer (6)": [6], "2 layer (6,3)": [6, 3]}
LM_ITERS = 5 if SMOKE else int(os.environ.get("LM_ITERS", 60))
LM_MAX_TRAIN_S = 300.0
MAX_TRAIN_ROWS = int(os.environ.get("MAX_TRAIN_ROWS", 20000 if SMOKE else 120000))
THRESHOLDS = np.linspace(0.05, 0.95, 91)

def best_threshold(p, y):
    best_t, best_f = 0.5, -1.0
    for t in THRESHOLDS:
        f = f1_score(y, (p > t).astype(int), zero_division=0)
        if f > best_f:
            best_t, best_f = float(t), f
    return best_t, best_f

def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan

def build_xy(version, cond):
    # kembalikan F (baris x fitur), y, split, sensor_id (0..3; -1 untuk rata-rata), t_index
    if version == "rata2":
        g = GEN_AVG[cond]
        F = lag_features(g["y"]); y = (g["m"] > 0).astype(int)
        keep = valid_row
        return F[keep], y[keep], split_of[keep], np.full(keep.sum(), -1), np.flatnonzero(keep)
    g = GEN_4S[cond]
    Fs, ys, sp, sid, ti = [], [], [], [], []
    for s in range(4):
        F = lag_features(g["Y"][:, s]); y = (g["M"][:, s] > 0).astype(int)
        Fs.append(F[valid_row]); ys.append(y[valid_row]); sp.append(split_of[valid_row])
        sid.append(np.full(valid_row.sum(), s)); ti.append(np.flatnonzero(valid_row))
    return (np.vstack(Fs), np.concatenate(ys), np.concatenate(sp),
            np.concatenate(sid), np.concatenate(ti))

def train_eval(version, cond, arch_name, hidden, seed):
    F, y, sp, sid, ti = build_xy(version, cond)
    tr = np.flatnonzero(sp == 0)
    if len(tr) > MAX_TRAIN_ROWS:
        tr = np.sort(np.random.default_rng(seed).choice(tr, MAX_TRAIN_ROWS, replace=False))
    scaler = StandardScaler().fit(F[tr])
    Z = scaler.transform(F)
    c0 = _time.process_time(); t0 = _time.perf_counter()
    res = lm_fit(Z[tr], np.eye(2)[y[tr]], Z.shape[1], hidden, 2, activation="tanh",
                 iters=LM_ITERS, seed=seed, max_train_s=LM_MAX_TRAIN_S)
    wall = _time.perf_counter() - t0; cpu = _time.process_time() - c0
    p = lm_predict_proba(res["theta"], res["template"], Z)[:, 1]
    va, te = sp == 1, sp == 2
    thr, f1_val = best_threshold(p[va], y[va])
    yhat = (p > thr).astype(int)
    row = {"versi": version, "skenario": CONDITIONS[cond][0], "kondisi": cond, "arsitektur": arch_name,
           "n_latih": len(tr), "n_uji": int(te.sum()), "fault_uji_%": round(100 * y[te].mean(), 2),
           "ambang": thr, "F1_validasi": round(f1_val, 4),
           "akurasi": accuracy_score(y[te], yhat[te]),
           "precision": precision_score(y[te], yhat[te], zero_division=0),
           "recall": recall_score(y[te], yhat[te], zero_division=0),
           "F1": f1_score(y[te], yhat[te], zero_division=0),
           "ROC_AUC": safe_auc(y[te], p[te]),
           "lm_iter": res["hist"]["iters"], "lm_status": res["hist"]["status"],
           "train_mse": res["train_mse"], "wall_s": wall, "cpu_s": cpu}
    per_sensor = []
    if version == "4sensor":
        for s in range(4):
            mk = te & (sid == s)
            yt, yh = y[mk], yhat[mk]
            neg = yt == 0
            per_sensor.append({"kondisi": cond, "skenario": CONDITIONS[cond][0], "arsitektur": arch_name,
                               "sensor": s + 1, "terinjeksi": int(s in GEN_4S[cond]["subset"]),
                               "n_fault_uji": int(yt.sum()),
                               "precision": precision_score(yt, yh, zero_division=0) if yt.sum() else np.nan,
                               "recall": recall_score(yt, yh, zero_division=0) if yt.sum() else np.nan,
                               "F1": f1_score(yt, yh, zero_division=0) if yt.sum() else np.nan,
                               "FPR": float(yh[neg].mean()) if neg.any() else np.nan})
    pred = {"t": ti, "sensor": sid, "split": sp, "label": y, "prob": p, "alarm": yhat, "thr": thr}
    return row, per_sensor, pred, res["hist"]


In [ ]:
RESULTS, PER_SENSOR, PREDS, HISTS = [], [], {}, {}
for version in ["rata2", "4sensor"]:
    for ci, cond in enumerate(CONDITIONS):
        for arch_name, hidden in ARCHS.items():
            row, ps, pred, hist = train_eval(version, cond, arch_name, hidden, seed=RANDOM_SEED + ci)
            RESULTS.append(row); PER_SENSOR.extend(ps)
            PREDS[(version, cond, arch_name)] = pred; HISTS[(version, cond, arch_name)] = hist
            print(f"[{time.time()-NOTEBOOK_START:7.0f}s] {version:7s} {row['skenario']} {cond:26s} {arch_name:14s} "
                  f"F1={row['F1']:.3f} AUC={row['ROC_AUC']:.3f} iter={row['lm_iter']} {row['lm_status']} "
                  f"wall={row['wall_s']:.1f}s", flush=True)

hasil = pd.DataFrame(RESULTS)
for c in ["akurasi", "precision", "recall", "F1", "ROC_AUC", "train_mse", "wall_s", "cpu_s"]:
    hasil[c] = hasil[c].astype(float).round(4)
export_df(hasil, "15_hasil_ann_lm")
per_sensor = pd.DataFrame(PER_SENSOR).round(4)
export_df(per_sensor, "15_hasil_ann_lm_per_sensor")


# §6 — Hasil

Tabel di bawah: F1 di data **uji** per skenario (rata-rata kondisi dalam satu
skenario), lalu rincian per kondisi. Ambang alarm dipilih di validasi.


In [ ]:
ringkas = (hasil.groupby(["versi", "arsitektur", "skenario"])[["precision", "recall", "F1", "ROC_AUC"]]
           .mean().round(3).reset_index())
export_df(ringkas, "15_hasil_ringkas_per_skenario")
display(ringkas.pivot_table(index=["versi", "arsitektur"], columns="skenario", values="F1"))
display(hasil[["versi", "skenario", "kondisi", "arsitektur", "fault_uji_%", "precision", "recall", "F1",
               "ROC_AUC", "lm_iter", "wall_s"]])

if len(per_sensor):
    ps = per_sensor[per_sensor["terinjeksi"] == 1]
    display(ps.groupby(["arsitektur", "sensor"])[["precision", "recall", "F1"]].mean().round(3))
    fpr = per_sensor[per_sensor["terinjeksi"] == 0].groupby("arsitektur")["FPR"].mean().round(4)
    print("FPR rata-rata pada sensor yang TIDAK disuntik fault:"); display(fpr)

# Kurva loss LM untuk satu kondisi per skenario
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, version in zip(axes, ["rata2", "4sensor"]):
    for cond in ["faulty", "drift", "bias+malfunc", "drift+bias+malfunc", "spike+bias+malfunc+drift"]:
        h = HISTS[(version, cond, "2 layer (6,3)")]
        ax.plot(h["loss"], label=f"{CONDITIONS[cond][0]} {cond}")
    ax.set_yscale("log"); ax.set_xlabel("iterasi LM"); ax.set_ylabel("MSE latih"); ax.set_title(version)
axes[0].legend(fontsize=7); fig.tight_layout()
fig.savefig(EXPORT_DIR / "15_kurva_loss_lm.png", dpi=90); plt.show()


## Prediksi per sampel (untuk sistem notifikasi)

Untuk tiap versi dan skenario, model 2 layer (6,3) diterapkan ke **seluruh**
data kondisi itu. Kolom `prob_fault` = probabilitas gangguan dari ANN-LM,
`alarm` = 1 bila `prob_fault` > `ambang` (dipilih di validasi), `split` =
bagian data (latih/validasi/uji — angka evaluasi yang jujur hanya dari `uji`).
Sistem notifikasi bisa langsung membaca kolom `alarm`, atau memakai
`prob_fault` dengan ambang/aturan persistensi sendiri (mis. alarm baru dikirim
kalau `alarm` = 1 selama beberapa sampel berturut-turut).


In [ ]:
PRED_ARCH = "2 layer (6,3)"
for version in ["rata2", "4sensor"]:
    for scen in SCENARIO_ORDER:
        parts = []
        for cond in [c for c, (s, _) in CONDITIONS.items() if s == scen]:
            pr = PREDS[(version, cond, PRED_ARCH)]
            part = pd.DataFrame({"timestamp": df["timestamp"].to_numpy()[pr["t"]], "skenario": scen,
                                 "kondisi": cond, "split": pd.Series(pr["split"]).map(SPLIT_NAME).to_numpy(),
                                 "label_fault": pr["label"], "prob_fault": np.round(pr["prob"], 5),
                                 "ambang": round(pr["thr"], 3), "alarm": pr["alarm"]})
            if version == "4sensor":
                part.insert(3, "sensor", pr["sensor"] + 1)
            parts.append(part)
        export_df(pd.concat(parts, ignore_index=True), f"15_prediksi_{version}_{scen}", gz=True)


In [ ]:
files = sorted(EXPORT_DIR.glob("15_*"))
daftar = pd.DataFrame({"file": [f.name for f in files],
                       "ukuran_MB": [round(f.stat().st_size / 1e6, 2) for f in files]})
display(daftar)
# baca balik satu file data supaya yakin bisa dipakai
_chk = pd.read_csv(EXPORT_DIR / "15_data_hipotetik_4sensor_S5.csv.gz")
assert {"label_s1", "jenis_s1", "label_fault", "sensor_rusak"} <= set(_chk.columns)
assert _chk["label_fault"].isin([0, 1]).all()
print(f"Selesai. Wall total {time.time()-NOTEBOOK_START:.0f} s, {len(files)} file di {EXPORT_DIR}/")
